# Submit a Command Job

Generate a taxi-data summarization script, run it on the configured Azure ML compute, and inspect its terminal status and downloaded summary response.

Set `AZUREML_COMPUTE_NAME` and `AZUREML_COMPUTE_IDENTITY_CLIENT_ID` in the suite's `.env`. Use the cluster identity's **Client ID**, not its Principal ID. This notebook does not read managed-identity resources. Run setup, preparation, and submission separately.

**Sources:** Adapted from this repository's command-job patterns and the [Azure ML single-step examples](https://github.com/Azure/azureml-examples/tree/main/sdk/python/jobs/single-step), MIT License.

In [ ]:
from pathlib import Path
from azure.identity import AzureCliCredential, ManagedIdentityCredential
from dotenv import dotenv_values
import ast
import json
import textwrap
from azure.ai.ml import Input, MLClient, Output, command
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import ManagedIdentityConfiguration

notebook_path = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "05_submit_command_job.ipynb"))
SUITE_ROOT = notebook_path.resolve().parents[1]
env = dotenv_values(SUITE_ROOT / ".env", encoding="utf-8-sig", interpolate=False)

SUBSCRIPTION_ID = (env.get("AZURE_SUBSCRIPTION_ID") or "").strip()
RESOURCE_GROUP = (env.get("AZURE_RESOURCE_GROUP") or "").strip()
WORKSPACE_NAME = (env.get("AZUREML_WORKSPACE_NAME") or "").strip()
COMPUTE_NAME = (env.get("AZUREML_COMPUTE_NAME") or "").strip()
COMPUTE_IDENTITY_CLIENT_ID = (env.get("AZUREML_COMPUTE_IDENTITY_CLIENT_ID") or "").strip()
if not all((
    SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME,
    COMPUTE_NAME, COMPUTE_IDENTITY_CLIENT_ID,
)):
    raise ValueError(f"Fill workspace coordinates, compute name, and identity Client ID in {SUITE_ROOT / '.env'}.")

use_managed_identity = (env.get("USE_MANAGED_IDENTITY", "true") or "").strip().lower()
if use_managed_identity not in {"true", "false"}:
    raise ValueError("USE_MANAGED_IDENTITY must be true or false in .env.")
credential = ManagedIdentityCredential(process_timeout=60) if use_managed_identity == "true" else AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"
EXPERIMENT_NAME = "workspace-readiness"
print(f"Workspace: {WORKSPACE_NAME}; compute: {COMPUTE_NAME}")

## Prepare the Job

Generate the existing summarization payload and construct the job using the configured cluster and Client ID. This cell does not upload or submit a job.

In [ ]:
generated_code_dir = SUITE_ROOT / "outputs/generated/foundations/command_job/code"
generated_code_dir.mkdir(parents=True, exist_ok=True)
(generated_code_dir / ".amlignore").write_text(
    "__pycache__/\n*.py[cod]\n", encoding="utf-8"
)
script_path = generated_code_dir / "summarize.py"
script_source = r'''
import argparse
import json
from pathlib import Path

import pandas as pd


def resolve_csv(path):
    path = Path(path)
    if path.is_file():
        return path
    matches = sorted(path.rglob("*.csv"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one CSV beneath {path}, found {len(matches)}")
    return matches[0]


parser = argparse.ArgumentParser(description="Summarize a taxi CSV")
parser.add_argument("--input-data", required=True)
parser.add_argument("--output-data", required=True)
args = parser.parse_args()
source_path = resolve_csv(args.input_data)
frame = pd.read_csv(source_path)
output_dir = Path(args.output_data)
output_dir.mkdir(parents=True, exist_ok=True)
summary = {
    "source_file": source_path.name,
    "row_count": len(frame),
    "column_count": len(frame.columns),
    "columns": frame.columns.tolist(),
}
(output_dir / "summary.json").write_text(
    json.dumps(summary, indent=2) + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, indent=2))
'''
script_source = textwrap.dedent(script_source).lstrip()
ast.parse(script_source, filename=str(script_path))
script_path.write_text(script_source, encoding="utf-8")
print(f"Generated command script: {script_path}")

job = command(
    display_name="Readiness taxi CSV summary",
    experiment_name=EXPERIMENT_NAME,
    code=str(generated_code_dir),
    command=(
        "python summarize.py "
        "--input-data ${{inputs.input_data}} "
        "--output-data ${{outputs.output_data}}"
    ),
    inputs={
        "input_data": Input(
            type=AssetTypes.URI_FILE,
            path=str(SAMPLE_ROOT / "data/taxi/raw/yellowTaxiData.csv"),
        )
    },
    outputs={
        "output_data": Output(type=AssetTypes.URI_FOLDER, mode="rw_mount")
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute=COMPUTE_NAME,
    identity=ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID),
    tags={"validation": "workspace-readiness", "operation": "command-job"},
)

print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")
print(f"Job prepared for cluster: {job.compute}; run the next cell to submit.")


## Submit the Job

Run this cell once to upload and execute the prepared job. It can incur compute and storage charges. Use the separate status and output cells to inspect the same run without resubmitting.

In [ ]:
submitted_job = ml_client.jobs.create_or_update(job)
print(f"Submitted job: {submitted_job.name}")

## Check Status

Rerun this read-only cell while the job is running. After a kernel restart, replace `submitted_job.name` with the quoted existing run name. Do not resubmit to retrieve results. This uses job status instead of the SDK log streamer, which fails with token-based datastore credentials.

In [ ]:
JOB_NAME = submitted_job.name
final_job = ml_client.jobs.get(JOB_NAME)
print(f"Job: {final_job.name}; status: {final_job.status}")
print(f"Studio: {final_job.studio_url}")

## Download the Completed Job's Response

Run after status is `Completed`. This downloads the named output and displays the actual summary JSON; it does not submit another job.

In [ ]:
import time

deadline = time.monotonic() + 20 * 60
final_job = ml_client.jobs.get(JOB_NAME)
while final_job.status not in {"Completed", "Failed", "Canceled", "NotResponding"}:
    if time.monotonic() >= deadline:
        raise TimeoutError(f"Timed out after 20 minutes waiting for job {JOB_NAME}; last status: {final_job.status}.")
    time.sleep(10)
    final_job = ml_client.jobs.get(JOB_NAME)
if final_job.status in {"Failed", "Canceled", "NotResponding"}:
    raise RuntimeError(f"Job ended with status {final_job.status}; inspect its Studio page.")
if final_job.status == "Completed":
    download_dir = SUITE_ROOT / "outputs/foundation_command_job" / final_job.name
    ml_client.jobs.download(
        final_job.name,
        output_name="output_data",
        download_path=download_dir,
    )
    summary_path = download_dir / "named-outputs/output_data/summary.json"
    if not summary_path.is_file():
        raise FileNotFoundError(f"Expected downloaded output at {summary_path}")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(summary)
    print(f"Downloaded summary: {summary_path}")


## Expected Result

The command job completes on the configured cluster and publishes a `summary.json` artifact in its output folder.

Next: `06_deploy_online_endpoint.ipynb`.